In [2]:
def parse_treeducken_file(filepath):
    host_newick = None
    parasite_newick = None
    associations = []
    sim_stats = {}

    in_host_block = False
    in_parasite_block = False
    in_distribution_block = False

    with open(filepath, 'r') as file:
        lines = file.readlines()

    for line in lines:
        stripped = line.strip()

        # Identify block starts
        if stripped.startswith("BEGIN HOST"):
            in_host_block = True
            continue
        elif stripped.startswith("BEGIN PARASITE"):
            in_parasite_block = True
            continue
        elif stripped.startswith("BEGIN DISTRIBUTION"):
            in_distribution_block = True
            continue

        # Identify block ends
        if stripped.startswith("ENDBLOCK"):
            in_host_block = in_parasite_block = in_distribution_block = False
            continue

        # Extract Newick strings
        if in_host_block and "TREE" in stripped:
            host_newick = stripped.split("=")[1].strip(" ;")

        elif in_parasite_block and "TREE" in stripped:
            parasite_newick = stripped.split("=")[1].strip(" ;")

        # Extract RANGE mappings
        elif in_distribution_block and ":" in stripped:
            parasite, host = map(str.strip, stripped.split(":"))
            associations.append((parasite, host))

    # Parse statistics from end lines (formatted as: Key [space] Value)
    stat_pattern = re.compile(r"^([A-Za-z_/]+[\w\s/]*?)\s+([-\d.eE]+|NaN)$")
    for line in lines[::-1]:  # reverse iterate to focus on end of file
        match = stat_pattern.match(line.strip())
        if match:
            key = match.group(1).strip().replace(" ", "_").replace("/", "_")
            val_str = match.group(2).strip()
            try:
                val = float(val_str) if val_str.lower() != "nan" else None
                sim_stats[key] = val
            except ValueError:
                print(f"Could not convert value to float for line: {line.strip()}")

    data = {
        "filename": os.path.basename(filepath),
        "host_newick": host_newick,
        "parasite_newick": parasite_newick,
        "associations": associations,
        "sim_stats": sim_stats
    }

    return data

def parse_all_treeducken_in_folder(folder_path):
    datasets = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".tgl"):
            full_path = os.path.join(folder_path, filename)
            try:
                data = parse_treeducken_file(full_path)
                datasets.append(data)
            except Exception as e:
                print(f"Error parsing {filename}: {e}")
    return datasets

In [3]:
import os 
import re
datasets_treeducken_cosp = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highCosp/Datasets')
datasets_treeducken_switch = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_highSwitch/Datasets')
datasets_treeducken_medium = parse_all_treeducken_in_folder('/Users/gabriele/synthetic_cophylo/generate_treeducken/generated_trees_medium/Datasets')

In [13]:
def compute_selected_event_counts(entry):
    stats = entry["sim_stats"]
    total = stats.get("Total_Events")
    if total is None:
        total = 0  # rule: if total is None → 0

    # Only these events must be multiplied
    target_events = [
        "Host_Spread_Switches",
        "Symbiont_Extinctions",
        "Symbiont_Speciations",
        "Host_Extinctions",
        "Host_Speciations",
        "Cospeciations",
    ]

    result = {}

    for event in target_events:
        freq = stats.get(event)

        if freq is None:
            result[event] = 0
        else:
            result[event] = int(round(freq * total))

    return result

In [14]:
combined_data = []

for d in datasets_treeducken_cosp:   # your list of dictionaries
    stats = d["sim_stats"]
    
    row = {
        "filename": d["filename"],
        "parasite_leaves": stats.get("Parasite_num_leaves", 0),
        "host_leaves": stats.get("Host_num_leaves", 0),
        "total_events": stats.get("Total_Events", 0),
    }

    # add detailed event counts
    row.update(compute_selected_event_counts(d))

    combined_data.append(row)
    
import pandas as pd

df_cosp_treeducken = pd.DataFrame(combined_data)
df_cosp_treeducken

,filename,parasite_leaves,host_leaves,total_events,Host_Spread_Switches,Symbiont_Extinctions,Symbiont_Speciations,Host_Extinctions,Host_Speciations,Cospeciations
0,Dataset399.tgl,37.0,38.0,76.0,0,16,10,13,11,26
1,Dataset428.tgl,32.0,34.0,69.0,1,17,10,8,13,20
2,Dataset400.tgl,41.0,35.0,87.0,0,29,13,11,8,26
3,Dataset366.tgl,49.0,57.0,101.0,0,21,12,12,21,35
4,Dataset372.tgl,28.0,26.0,53.0,0,14,6,8,5,20
...,...,...,...,...,...,...,...,...,...,...
495,Dataset357.tgl,68.0,62.0,126.0,0,27,22,16,16,45
496,Dataset343.tgl,35.0,26.0,62.0,0,18,12,7,3,22
497,Dataset425.tgl,25.0,27.0,54.0,0,11,11,6,14,12
498,Dataset394.tgl,12.0,12.0,25.0,0,6,2,6,2,9


In [15]:
combined_data = []

for d in datasets_treeducken_switch:   # your list of dictionaries
    stats = d["sim_stats"]
    
    row = {
        "filename": d["filename"],
        "parasite_leaves": stats.get("Parasite_num_leaves", 0),
        "host_leaves": stats.get("Host_num_leaves", 0),
        "total_events": stats.get("Total_Events", 0),
    }

    # add detailed event counts
    row.update(compute_selected_event_counts(d))

    combined_data.append(row)
    
import pandas as pd

df_switch_treeducken = pd.DataFrame(combined_data)
df_switch_treeducken

,filename,parasite_leaves,host_leaves,total_events,Host_Spread_Switches,Symbiont_Extinctions,Symbiont_Speciations,Host_Extinctions,Host_Speciations,Cospeciations
0,Dataset399.tgl,28.0,9.0,35.0,6,10,9,2,7,1
1,Dataset428.tgl,36.0,19.0,56.0,17,10,9,2,18,0
2,Dataset400.tgl,32.0,12.0,39.0,11,7,9,1,9,2
3,Dataset366.tgl,20.0,7.0,28.0,4,3,14,1,6,0
4,Dataset372.tgl,31.0,6.0,43.0,5,18,12,3,4,1
...,...,...,...,...,...,...,...,...,...,...
495,Dataset357.tgl,18.0,6.0,26.0,2,7,11,1,2,3
496,Dataset343.tgl,51.0,18.0,75.0,11,17,23,7,14,3
497,Dataset425.tgl,20.0,6.0,32.0,6,11,7,3,3,2
498,Dataset394.tgl,68.0,24.0,88.0,16,23,24,2,20,3


In [16]:
combined_data = []

for d in datasets_treeducken_medium:   # your list of dictionaries
    stats = d["sim_stats"]
    
    row = {
        "filename": d["filename"],
        "parasite_leaves": stats.get("Parasite_num_leaves", 0),
        "host_leaves": stats.get("Host_num_leaves", 0),
        "total_events": stats.get("Total_Events", 0),
    }

    # add detailed event counts
    row.update(compute_selected_event_counts(d))

    combined_data.append(row)
    
import pandas as pd

df_medium_treeducken = pd.DataFrame(combined_data)
df_medium_treeducken

,filename,parasite_leaves,host_leaves,total_events,Host_Spread_Switches,Symbiont_Extinctions,Symbiont_Speciations,Host_Extinctions,Host_Speciations,Cospeciations
0,Dataset399.tgl,19.0,7.0,29.0,4,9,8,2,2,4
1,Dataset428.tgl,15.0,14.0,26.0,0,4,8,1,8,5
2,Dataset400.tgl,8.0,12.0,17.0,0,2,3,1,8,3
3,Dataset366.tgl,15.0,8.0,24.0,3,7,5,2,3,4
4,Dataset372.tgl,35.0,25.0,61.0,9,10,13,5,19,5
...,...,...,...,...,...,...,...,...,...,...
495,Dataset357.tgl,21.0,14.0,32.0,5,6,5,3,7,6
496,Dataset343.tgl,26.0,21.0,53.0,2,17,9,5,10,10
497,Dataset425.tgl,13.0,7.0,27.0,0,9,7,5,3,3
498,Dataset394.tgl,36.0,29.0,69.0,7,18,9,7,15,13
